
# Autocallable Payoff Playground

This notebook is a **didactic sandbox** to visualize and tweak a simple autocallable payoff.

**What's inside**
- Parameters (barriers, coupons, tenor, etc.)
- Payoff function (pedagogical; not a pricing model)
- Payoff plot vs terminal performance
- Comparison vs direct equity (with a toy dividend yield) and bond
- Sample scenarios table
- Decomposition into simple building blocks (bond + short knock-in put + digital call)

> ⚠️ This is **not** a pricing notebook. It ignores path-dependence, greeks, discounting, smiles, and calendars.


In [ ]:

# === Parameters ===
N = 100.0     # nominal
S0 = 100.0    # initial level of underlying
Ba = 1.00     # autocall barrier as a fraction of S0 (1.00 = 100%)
Bp = 0.60     # protection barrier at maturity (e.g., 0.60 = 60%)
c  = 0.10     # coupon per observation period (10%)
years = 5     # maximum tenor in years
div_yield = 0.02   # toy dividend yield for equity comparison (per year)

# Terminal-performance grid (0..180%)
r_min, r_max, r_step = 0.0, 1.8, 0.0025


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Display settings
%matplotlib inline


In [ ]:

def payoff_autocall_terminal(ST_ratio: float) -> float:
    """
    Didactic payoff vs terminal level ratio R = S_T / S0.
    Uses geometric interpolation for intermediate observation levels:
      S_ti = S0 * R^(ti/T)
    Rule set (no coupon memory):
      - If exists i with S_ti >= Ba*S0, auto-call at earliest i with payoff N*(1 + c*i).
      - Else at maturity:
            if R >= Bp: return N
            else: return N * R  (linear downside)
    """
    R = max(0.0, ST_ratio)
    obs_times = np.arange(1, years+1)
    path_ratios = R ** (obs_times / years)
    ac_idx = None
    for i, r in zip(obs_times, path_ratios):
        if r >= Ba:
            ac_idx = i
            break
    if ac_idx is not None:
        return N * (1.0 + c * ac_idx)
    return N if R >= Bp else N * R


In [ ]:

R_grid = np.arange(r_min, r_max + r_step, r_step)
payoffs = np.array([payoff_autocall_terminal(r) for r in R_grid])
underlying_linear = N * R_grid
bond_line = np.full_like(R_grid, N)

plt.figure(figsize=(8,5))
plt.plot(R_grid*100, payoffs, label="Autocallable (didactic)")
plt.plot(R_grid*100, underlying_linear, linestyle="--", label="Underlying (N×S_T/S0)")
plt.axvline(Ba*100, linestyle=":", label=f"Autocall barrier ({Ba:.0%})")
plt.axvline(Bp*100, linestyle=":", label=f"Protection barrier ({Bp:.0%})")
plt.xlabel("Terminal performance S_T / S0 (%)")
plt.ylabel("Redemption at call/maturity")
plt.title(f"Autocallable payoff vs terminal performance ({years}Y, coupon {c:.0%})")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

equity_with_div = N * R_grid * (1 + div_yield * years)

plt.figure(figsize=(8,5))
plt.plot(R_grid*100, payoffs, label=f"Autocallable ({c:.0%}/period, Bp={Bp:.0%})", linewidth=2)
plt.plot(R_grid*100, equity_with_div, "--", label=f"Equity held (+{div_yield:.0%}/y dividend)")
plt.plot(R_grid*100, bond_line, ":", label="Bond (capital)")
plt.axvline(Bp*100, linestyle=":", color="red")
plt.axvline(Ba*100, linestyle=":", color="gray")
plt.xlabel("Terminal performance S_T / S0 (%)")
plt.ylabel("Final redemption (% of nominal)")
plt.title("Autocallable vs Equity (with dividends) vs Bond")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

def earliest_autocall_year(ST_ratio: float):
    R = max(0.0, ST_ratio)
    obs_times = np.arange(1, years+1)
    path_ratios = R ** (obs_times / years)
    for i, r in zip(obs_times, path_ratios):
        if r >= Ba:
            return int(i)
    return None

sample_ST = [0.5, 0.6, 0.95, 1.0, 1.02, 1.5]
rows = []
for r in sample_ST:
    rows.append({
        "S_T / S0": f"{r:,.0%}",
        "Earliest AC year": earliest_autocall_year(r) if earliest_autocall_year(r) is not None else "No AC",
        "Redemption": round(payoff_autocall_terminal(r), 2)
    })
pd.DataFrame(rows)


In [ ]:

def comp_bond(R):        return N
def comp_short_ki_put(R):  return -N * (1.0 - R) * (1.0 if R < Bp else 0.0)
def comp_digital_call(R):  return N * c * (1.0 if R >= 1.0 else 0.0)

bond_curve     = np.array([comp_bond(r) for r in R_grid])
ki_put_curve   = np.array([comp_short_ki_put(r) for r in R_grid])
digital_curve  = np.array([comp_digital_call(r) for r in R_grid])
sum_components = bond_curve + ki_put_curve + digital_curve

plt.figure(figsize=(9,6))
plt.plot(R_grid*100, bond_curve, label="(1) Bond (N)")
plt.plot(R_grid*100, ki_put_curve, label="(2) Short KI Put (barrier)")
plt.plot(R_grid*100, digital_curve, label="(3) Digital Call @100% (coupon)")
plt.plot(R_grid*100, sum_components, linewidth=3, label="Sum of components")
plt.plot(R_grid*100, np.array([payoff_autocall_terminal(r) for r in R_grid]), "--", linewidth=2, label="Target payoff")
plt.axvline(Bp*100, linestyle=":", color="orange")
plt.axvline(100, linestyle=":", color="gray")
plt.xlabel("Terminal performance S_T / S0 (%)")
plt.ylabel("Payoff (% of nominal)")
plt.title("Didactic decomposition of an autocallable")
plt.legend()
plt.tight_layout()
plt.show()
